# Tema 5 - Realimentación

**Electrónica General - 2º GIERM**

---

## Objetivos de aprendizaje

- Comprender el concepto de realimentación negativa y su diagrama de bloques
- Derivar la ganancia en lazo cerrado $A_{CL}$ y entender el papel de $A \cdot \beta$
- Cuantificar los beneficios: desensibilización, aumento de ancho de banda, reducción de distorsión
- Conocer las 4 topologías de realimentación y su efecto sobre impedancias
- Analizar la estabilidad de sistemas realimentados (margen de fase y ganancia)
- Resolver problemas numéricos de diseño con realimentación

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.lines import Line2D
import schemdraw
import schemdraw.elements as elm

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 13
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['figure.dpi'] = 100

COLOR_PRINCIPAL = '#2171b5'   # azul - curvas principales
COLOR_RECTA = '#cb181d'       # rojo - rectas de carga, límites
COLOR_PUNTO = '#238b45'       # verde - puntos de operación
COLOR_BETA = '#6a3d9a'        # púrpura - red de realimentación
COLOR_CL = '#ff7f00'          # naranja - lazo cerrado

print('Configuración lista.')

---

# Formulario de Realimentación

## Ganancia en lazo cerrado

$$\boxed{A_{CL} = \frac{A}{1 + A\beta}}$$

Para $A\beta \gg 1$:

$$\boxed{A_{CL} \approx \frac{1}{\beta}}$$

## Ganancia de lazo

$$\boxed{A_L = -A\beta} \qquad \text{(loop gain)}$$

## Desensibilización de la ganancia

$$\boxed{\frac{dA_{CL}}{A_{CL}} \approx \frac{1}{1 + A\beta} \cdot \frac{dA}{A}}$$

$$\boxed{\Delta A_{CL} \approx \frac{\Delta A}{(1 + A\beta)^2}}$$

## Aumento de ancho de banda

$$\boxed{\omega_{CL,p1} = (1 + A_0 \beta)\,\omega_{p1}}$$

$$\boxed{GBW_{CL} = GBW_{OL}} \qquad \text{(se conserva el producto ganancia-ancho de banda)}$$

## Reducción de distorsión (misma entrada)

$$\boxed{HD_{CL,i} \approx \frac{HD_i}{(1 + a_1 \beta)^i}}$$

## Reducción de distorsión (misma salida)

$$\boxed{HD_{CL,i} \approx \frac{HD_i}{1 + a_1 \beta}}$$

## 4 Topologías de realimentación

| Topología | Muestreo | Mezcla | Efecto en $Z_{in}$ | Efecto en $Z_{out}$ |
|-----------|----------|--------|---------------------|----------------------|
| **Serie-Paralelo** (tensión-tensión) | Tensión (paralelo) | Serie | $Z_{in} \uparrow \times (1+A\beta)$ | $Z_{out} \downarrow \div (1+A\beta)$ |
| **Serie-Serie** (corriente-tensión) | Corriente (serie) | Serie | $Z_{in} \uparrow \times (1+A\beta)$ | $Z_{out} \uparrow \times (1+A\beta)$ |
| **Paralelo-Paralelo** (tensión-corriente) | Tensión (paralelo) | Paralelo | $Z_{in} \downarrow \div (1+A\beta)$ | $Z_{out} \downarrow \div (1+A\beta)$ |
| **Paralelo-Serie** (corriente-corriente) | Corriente (serie) | Paralelo | $Z_{in} \downarrow \div (1+A\beta)$ | $Z_{out} \uparrow \times (1+A\beta)$ |

---

# 1. Introducción a la Realimentación

## 1.1 ¿Qué es la realimentación?

La **realimentación** (feedback) consiste en tomar una muestra de la señal de salida de un sistema y devolverla a la entrada para compararla con la señal deseada. Es uno de los conceptos más importantes en ingeniería electrónica y de control.

### Analógico vs. Digital

| Aspecto | Mundo analógico | Mundo digital |
|---------|----------------|---------------|
| Ejemplo | Amplificador con resistencias de retroalimentación | PLL, convertidores $\Sigma\Delta$ |
| Señal de error | Continua en el tiempo | Muestreada/cuantizada |
| Análisis | Función de transferencia en $s$ | Función de transferencia en $z$ |

### ¿Por qué realimentación negativa?

La **realimentación negativa** sacrifica ganancia a cambio de:

1. **Menor sensibilidad** a variaciones de los componentes activos
2. **Mayor ancho de banda**
3. **Menor distorsión**
4. **Control de impedancias** de entrada y salida
5. **Mayor linealidad** del sistema

> **Nota:** La realimentación *positiva* se usa en osciladores y biestables, pero no es el foco de este tema.

In [ ]:
# Diagrama de bloques de un sistema realimentado
fig, ax = plt.subplots(1, 1, figsize=(12, 5))
ax.set_xlim(-1, 11)
ax.set_ylim(-2, 4)
ax.set_aspect('equal')
ax.axis('off')

# Nodo sumador (círculo)
circle = plt.Circle((2, 2), 0.4, fill=False, edgecolor='black', linewidth=2)
ax.add_patch(circle)
ax.text(2, 2, r'$\Sigma$', ha='center', va='center', fontsize=16, fontweight='bold')
ax.text(1.65, 2.35, '+', fontsize=12, color=COLOR_PUNTO, fontweight='bold')
ax.text(1.65, 1.55, r'$-$', fontsize=14, color=COLOR_RECTA, fontweight='bold')

# Bloque A (amplificador)
rect_A = FancyBboxPatch((4, 1.2), 2.5, 1.6, boxstyle="round,pad=0.1",
                         facecolor='#d4e6f1', edgecolor=COLOR_PRINCIPAL, linewidth=2)
ax.add_patch(rect_A)
ax.text(5.25, 2, r'$A$', ha='center', va='center', fontsize=22, fontweight='bold',
        color=COLOR_PRINCIPAL)
ax.text(5.25, 1.45, '(ganancia OL)', ha='center', va='center', fontsize=9, color='gray')

# Bloque beta (red de realimentación)
rect_beta = FancyBboxPatch((4, -1.3), 2.5, 1.2, boxstyle="round,pad=0.1",
                            facecolor='#e8daef', edgecolor=COLOR_BETA, linewidth=2)
ax.add_patch(rect_beta)
ax.text(5.25, -0.7, r'$\beta$', ha='center', va='center', fontsize=22, fontweight='bold',
        color=COLOR_BETA)
ax.text(5.25, -1.15, '(realimentación)', ha='center', va='center', fontsize=9, color='gray')

# Flechas y conexiones
arrow_kw = dict(arrowstyle='->', mutation_scale=20, linewidth=2, color='black')

# x -> sumador
ax.annotate('', xy=(1.6, 2), xytext=(0, 2), arrowprops=arrow_kw)
ax.text(0.7, 2.3, r'$x$', fontsize=16, fontweight='bold')

# sumador -> A
ax.annotate('', xy=(4, 2), xytext=(2.4, 2), arrowprops=arrow_kw)
ax.text(3.1, 2.3, r'$u = x - \beta y$', fontsize=11, color='gray')

# A -> salida
ax.annotate('', xy=(9.5, 2), xytext=(6.5, 2), arrowprops=arrow_kw)
ax.text(9.6, 2, r'$y = A \cdot u$', fontsize=14, fontweight='bold')

# Punto de bifurcación
ax.plot(8, 2, 'ko', markersize=6)

# salida -> beta (bajada)
ax.plot([8, 8], [2, -0.7], 'k-', linewidth=2)
ax.annotate('', xy=(6.5, -0.7), xytext=(8, -0.7), arrowprops=arrow_kw)

# beta -> sumador (subida)
ax.plot([4, 2], [-0.7, -0.7], 'k-', linewidth=2)
ax.plot([2, 2], [-0.7, 1.6], 'k-', linewidth=2)
ax.annotate('', xy=(2, 1.6), xytext=(2, 0.5),
            arrowprops=dict(arrowstyle='->', mutation_scale=15, linewidth=2, color=COLOR_RECTA))
ax.text(0.8, -0.5, r'$\beta \cdot y$', fontsize=14, color=COLOR_BETA, fontweight='bold')

# Título
ax.set_title('Diagrama de Bloques - Sistema con Realimentación Negativa',
             fontsize=15, fontweight='bold', pad=15)

# Ecuación resultado
ax.text(5.25, -2, r'$A_{CL} = \frac{A}{1 + A\beta}$', fontsize=18,
        ha='center', va='center',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#ffffcc', edgecolor='gray'))

plt.tight_layout()
plt.show()

---

## 2. Realimentación ideal - Derivación de $A_{CL}$

### 2.1 Ecuaciones del sistema

Del diagrama de bloques:

$$y = A \cdot u$$

$$u = x - \beta \cdot y$$

Sustituyendo:

$$y = A(x - \beta y) = Ax - A\beta y$$

$$y + A\beta y = Ax$$

$$y(1 + A\beta) = Ax$$

$$\boxed{A_{CL} = \frac{y}{x} = \frac{A}{1 + A\beta}}$$

### 2.2 Interpretación

- **$A$**: Ganancia en lazo abierto (open loop, OL)
- **$\beta$**: Factor de realimentación
- **$A\beta$**: Ganancia de lazo (loop gain) - es la magnitud clave
- **$1 + A\beta$**: Factor de realimentación (siempre $> 1$ para realimentación negativa)

### 2.3 Caso límite: $A\beta \gg 1$

$$A_{CL} = \frac{A}{1 + A\beta} \approx \frac{A}{A\beta} = \frac{1}{\beta}$$

> **Resultado fundamental:** Cuando la ganancia de lazo es grande, la ganancia en lazo cerrado **depende solo de la red de realimentación** $\beta$, que normalmente se construye con componentes pasivos (resistencias) muy estables.

In [ ]:
# Ganancia en lazo cerrado vs ganancia en lazo abierto
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

beta = 0.01  # beta = 1/100 -> ganancia ideal = 100
A_values = np.logspace(0, 5, 500)
A_CL = A_values / (1 + A_values * beta)
A_ideal = 1 / beta

# Gráfica lineal
ax1.semilogx(A_values, A_CL, color=COLOR_CL, linewidth=2.5, label=r'$A_{CL} = A/(1+A\beta)$')
ax1.axhline(y=A_ideal, color=COLOR_RECTA, linestyle='--', linewidth=1.5,
            label=rf'$1/\beta = {A_ideal:.0f}$')
ax1.axhline(y=A_ideal*0.99, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax1.fill_between(A_values, A_ideal*0.99, A_ideal*1.01, alpha=0.1, color=COLOR_PUNTO)
ax1.set_xlabel(r'Ganancia en lazo abierto $A$')
ax1.set_ylabel(r'Ganancia en lazo cerrado $A_{CL}$')
ax1.set_title(rf'$A_{{CL}}$ vs $A$ ($\beta = {beta}$)')
ax1.legend(fontsize=11)
ax1.set_ylim(0, 120)
ax1.grid(True, alpha=0.3)

# Mostrar puntos clave
for A_val in [10, 100, 1000, 10000, 100000]:
    acl = A_val / (1 + A_val * beta)
    ax1.plot(A_val, acl, 'o', color=COLOR_PRINCIPAL, markersize=6)
    ax1.annotate(f'{acl:.1f}', xy=(A_val, acl), xytext=(5, -15),
                 textcoords='offset points', fontsize=9)

# Error relativo
error_rel = np.abs(A_CL - A_ideal) / A_ideal * 100
ax2.loglog(A_values, error_rel, color=COLOR_RECTA, linewidth=2.5)
ax2.set_xlabel(r'Ganancia en lazo abierto $A$')
ax2.set_ylabel('Error relativo (%)')
ax2.set_title(r'Error respecto a $1/\beta$')
ax2.axhline(y=1, color='gray', linestyle='--', linewidth=1, label='1% error')
ax2.axhline(y=0.1, color='gray', linestyle=':', linewidth=1, label='0.1% error')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Ejemplo numérico (\u03b2 = {beta}):")
print(f"{'A':>10} {'A\u03b2':>10} {'A_CL':>10} {'1/\u03b2':>10} {'Error(%)':>10}")
print("-"*55)
for A_val in [10, 100, 1000, 10000, 100000]:
    ab = A_val * beta
    acl = A_val / (1 + ab)
    err = abs(acl - A_ideal)/A_ideal * 100
    print(f"{A_val:10.0f} {ab:10.1f} {acl:10.2f} {A_ideal:10.0f} {err:10.2f}")

---

## 3. Desensibilización de la Ganancia

### 3.1 Derivación

Si la ganancia $A$ varía una cantidad $dA$, ¿cuánto varía $A_{CL}$?

$$A_{CL} = \frac{A}{1 + A\beta}$$

Derivando respecto a $A$:

$$\frac{dA_{CL}}{dA} = \frac{(1+A\beta) - A\beta}{(1+A\beta)^2} = \frac{1}{(1+A\beta)^2}$$

Por tanto:

$$dA_{CL} = \frac{dA}{(1+A\beta)^2}$$

En forma de **sensibilidad relativa**:

$$\frac{dA_{CL}}{A_{CL}} = \frac{dA}{(1+A\beta)^2} \cdot \frac{1+A\beta}{A} = \frac{1}{1+A\beta} \cdot \frac{dA}{A}$$

$$\boxed{\frac{dA_{CL}/A_{CL}}{dA/A} = \frac{1}{1+A\beta}}$$

### 3.2 Ejemplo numérico

Si $A\beta = 50$:

$$\frac{dA_{CL}}{dA} = \frac{1}{(1+50)^2} = \frac{1}{2601}$$

Una variación absoluta en $A$ se reduce **2601 veces** en $A_{CL}$.

La variación relativa se reduce $1+A\beta = 51$ veces.

---

## 4. Aumento del Ancho de Banda

### 4.1 Amplificador con un polo dominante

Si el amplificador en lazo abierto tiene un polo dominante:

$$A(s) = \frac{A_0}{1 + s/\omega_{p1}}$$

donde $A_0$ es la ganancia DC y $\omega_{p1}$ es la frecuencia del polo.

### 4.2 Ganancia en lazo cerrado

$$A_{CL}(s) = \frac{A(s)}{1 + A(s)\beta} = \frac{\frac{A_0}{1 + s/\omega_{p1}}}{1 + \frac{A_0 \beta}{1 + s/\omega_{p1}}}$$

Simplificando:

$$A_{CL}(s) = \frac{A_0}{1 + A_0\beta + s/\omega_{p1}} = \frac{A_0/(1+A_0\beta)}{1 + s/[(1+A_0\beta)\omega_{p1}]}$$

$$\boxed{A_{CL}(s) = \frac{A_{CL,0}}{1 + s/\omega_{CL,p1}}}$$

donde:

$$A_{CL,0} = \frac{A_0}{1 + A_0\beta}, \qquad \omega_{CL,p1} = (1 + A_0\beta)\,\omega_{p1}$$

### 4.3 Conservación del GBW

$$GBW_{OL} = A_0 \cdot \omega_{p1}$$

$$GBW_{CL} = A_{CL,0} \cdot \omega_{CL,p1} = \frac{A_0}{1+A_0\beta} \cdot (1+A_0\beta)\omega_{p1} = A_0 \cdot \omega_{p1}$$

$$\boxed{GBW_{CL} = GBW_{OL}}$$

> **Conclusión:** La realimentación negativa reduce la ganancia pero aumenta el ancho de banda en la misma proporción. El producto ganancia-ancho de banda se conserva.

In [ ]:
# Diagrama de Bode: lazo abierto vs lazo cerrado
A0 = 10000      # ganancia DC lazo abierto
fp1 = 1e3       # polo a 1 kHz
beta = 0.01     # factor de realimentación

wp1 = 2 * np.pi * fp1
f = np.logspace(1, 8, 1000)
w = 2 * np.pi * f
s = 1j * w

# Lazo abierto
A_OL = A0 / (1 + s / wp1)

# Lazo cerrado
A0_CL = A0 / (1 + A0 * beta)
wp1_CL = (1 + A0 * beta) * wp1
A_CL_s = A0_CL / (1 + s / wp1_CL)

# Loop gain
A_loop = A_OL * beta

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# Magnitud
ax1.semilogx(f, 20*np.log10(np.abs(A_OL)), color=COLOR_PRINCIPAL, linewidth=2.5,
             label=rf'$|A_{{OL}}|$ ($A_0={A0}$, $f_{{p1}}={fp1/1e3:.0f}$ kHz)')
ax1.semilogx(f, 20*np.log10(np.abs(A_CL_s)), color=COLOR_CL, linewidth=2.5,
             label=rf'$|A_{{CL}}|$ ($A_{{CL,0}}={A0_CL:.0f}$, $f_{{CL,p1}}={wp1_CL/(2*np.pi)/1e3:.0f}$ kHz)')
ax1.semilogx(f, 20*np.log10(np.abs(A_loop)), color=COLOR_BETA, linewidth=2,
             linestyle='--', label=rf'$|A\beta|$ ($\beta={beta}$)')
ax1.axhline(y=0, color='gray', linewidth=0.8, linestyle=':')

# Marcar BW
ax1.axvline(x=fp1, color=COLOR_PRINCIPAL, linestyle=':', alpha=0.5, linewidth=1.5)
ax1.axvline(x=wp1_CL/(2*np.pi), color=COLOR_CL, linestyle=':', alpha=0.5, linewidth=1.5)

# Anotar GBW
gbw = A0 * fp1
ax1.annotate(f'GBW = {gbw/1e6:.0f} MHz\n(se conserva)',
             xy=(gbw*0.3, 5), fontsize=11, color='gray',
             bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='gray'))

# Anotar anchos de banda
ax1.annotate(f'BW_OL = {fp1/1e3:.0f} kHz', xy=(fp1, 20*np.log10(A0)-3),
             xytext=(fp1*3, 20*np.log10(A0)-3), fontsize=10,
             arrowprops=dict(arrowstyle='->', color=COLOR_PRINCIPAL),
             color=COLOR_PRINCIPAL)
ax1.annotate(f'BW_CL = {wp1_CL/(2*np.pi)/1e6:.1f} MHz', xy=(wp1_CL/(2*np.pi), 20*np.log10(A0_CL)-3),
             xytext=(wp1_CL/(2*np.pi)*3, 20*np.log10(A0_CL)-3), fontsize=10,
             arrowprops=dict(arrowstyle='->', color=COLOR_CL),
             color=COLOR_CL)

ax1.set_ylabel('Magnitud (dB)')
ax1.set_title('Diagrama de Bode - Efecto de la Realimentación en el Ancho de Banda')
ax1.legend(fontsize=10, loc='upper right')
ax1.set_ylim(-20, 90)
ax1.grid(True, which='both', alpha=0.3)

# Fase
ax2.semilogx(f, np.degrees(np.angle(A_OL)), color=COLOR_PRINCIPAL, linewidth=2.5,
             label=r'$\angle A_{OL}$')
ax2.semilogx(f, np.degrees(np.angle(A_CL_s)), color=COLOR_CL, linewidth=2.5,
             label=r'$\angle A_{CL}$')
ax2.set_xlabel('Frecuencia (Hz)')
ax2.set_ylabel('Fase (grados)')
ax2.legend(fontsize=10)
ax2.set_ylim(-100, 10)
ax2.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

# Tabla resumen
print("\nComparación Lazo Abierto vs Lazo Cerrado:")
print(f"{'Parámetro':<25} {'Lazo Abierto':>15} {'Lazo Cerrado':>15} {'Factor':>10}")
print("-"*70)
print(f"{'Ganancia DC':<25} {A0:>15.0f} {A0_CL:>15.2f} {'\u00f7'+str(1+int(A0*beta)):>10}")
print(f"{'Ancho de banda':<25} {fp1:>12.0f} Hz {wp1_CL/(2*np.pi):>12.0f} Hz {'\u00d7'+str(1+int(A0*beta)):>10}")
print(f"{'GBW':<25} {A0*fp1:>12.1e} Hz {A0_CL*wp1_CL/(2*np.pi):>12.1e} Hz {'=':>10}")

---

## 5. Reducción de Distorsión

### 5.1 Modelo de distorsión

Un amplificador no lineal tiene una salida de la forma:

$$y = a_1 x + a_2 x^2 + a_3 x^3 + \cdots$$

donde $a_1$ es la ganancia lineal y los términos $a_i$ con $i > 1$ generan **distorsión armónica**.

El **coeficiente de distorsión armónica** de orden $i$ es:

$$HD_i = \frac{|a_i|}{|a_1|} \cdot A^{i-1}_{entrada}$$

### 5.2 Con realimentación (misma señal de entrada)

Al aplicar realimentación negativa, la distorsión se reduce drásticamente:

$$\boxed{HD_{CL,i} \approx \frac{HD_i}{(1 + a_1 \beta)^i}}$$

Para el segundo armónico ($i=2$): se reduce como $(1+a_1\beta)^2$.

### 5.3 Con realimentación (misma señal de salida)

Si se compara a **igual nivel de salida** (aumentando la entrada para compensar la pérdida de ganancia):

$$\boxed{HD_{CL,i} \approx \frac{HD_i}{1 + a_1 \beta}}$$

La mejora es menor pero aún significativa.

> **Importante:** La reducción de distorsión "a misma entrada" es más agresiva (potencia $i$), pero la comparación "a misma salida" es más realista en la práctica.

---

## 6. Las 4 Topologías de Realimentación

La forma en que se **muestrea** la salida y se **mezcla** con la entrada define 4 topologías posibles:

### 6.1 Serie-Paralelo (Tensión-Tensión)

- **Muestreo:** en paralelo con la salida (tensión)
- **Mezcla:** en serie con la entrada (tensión)
- **$A$ es una ganancia de tensión:** $A_v = v_o / v_i$
- **Efecto:**
  - $Z_{in,CL} = Z_{in} \cdot (1 + A\beta)$ (aumenta)
  - $Z_{out,CL} = Z_{out} / (1 + A\beta)$ (disminuye)
- **Ejemplo típico:** Amplificador operacional no inversor

### 6.2 Serie-Serie (Corriente-Tensión)

- **Muestreo:** en serie con la salida (corriente)
- **Mezcla:** en serie con la entrada (tensión)
- **$A$ es una transconductancia:** $G_m = i_o / v_i$
- **Efecto:**
  - $Z_{in,CL} = Z_{in} \cdot (1 + A\beta)$ (aumenta)
  - $Z_{out,CL} = Z_{out} \cdot (1 + A\beta)$ (aumenta)
- **Ejemplo típico:** Par diferencial con resistencia de degeneración

### 6.3 Paralelo-Paralelo (Tensión-Corriente)

- **Muestreo:** en paralelo con la salida (tensión)
- **Mezcla:** en paralelo con la entrada (corriente)
- **$A$ es una transimpedancia:** $R_m = v_o / i_i$
- **Efecto:**
  - $Z_{in,CL} = Z_{in} / (1 + A\beta)$ (disminuye)
  - $Z_{out,CL} = Z_{out} / (1 + A\beta)$ (disminuye)
- **Ejemplo típico:** Amplificador de transimpedancia

### 6.4 Paralelo-Serie (Corriente-Corriente)

- **Muestreo:** en serie con la salida (corriente)
- **Mezcla:** en paralelo con la entrada (corriente)
- **$A$ es una ganancia de corriente:** $A_i = i_o / i_i$
- **Efecto:**
  - $Z_{in,CL} = Z_{in} / (1 + A\beta)$ (disminuye)
  - $Z_{out,CL} = Z_{out} \cdot (1 + A\beta)$ (aumenta)
- **Ejemplo típico:** Espejo de corriente con realimentación

In [ ]:
# Resumen visual de las 4 topologías
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

topologies = [
    {'name': 'Serie-Paralelo\n(Tensión-Tensión)',
     'muestreo': 'Tensión\n(paralelo)', 'mezcla': 'Tensión\n(serie)',
     'A_type': r'$A_v = v_o/v_i$',
     'Zin': r'$Z_{in} \times (1+A\beta)$', 'Zout': r'$Z_{out} \div (1+A\beta)$',
     'color_in': COLOR_PUNTO, 'color_out': COLOR_RECTA},
    {'name': 'Serie-Serie\n(Corriente-Tensión)',
     'muestreo': 'Corriente\n(serie)', 'mezcla': 'Tensión\n(serie)',
     'A_type': r'$G_m = i_o/v_i$',
     'Zin': r'$Z_{in} \times (1+A\beta)$', 'Zout': r'$Z_{out} \times (1+A\beta)$',
     'color_in': COLOR_PUNTO, 'color_out': COLOR_PUNTO},
    {'name': 'Paralelo-Paralelo\n(Tensión-Corriente)',
     'muestreo': 'Tensión\n(paralelo)', 'mezcla': 'Corriente\n(paralelo)',
     'A_type': r'$R_m = v_o/i_i$',
     'Zin': r'$Z_{in} \div (1+A\beta)$', 'Zout': r'$Z_{out} \div (1+A\beta)$',
     'color_in': COLOR_RECTA, 'color_out': COLOR_RECTA},
    {'name': 'Paralelo-Serie\n(Corriente-Corriente)',
     'muestreo': 'Corriente\n(serie)', 'mezcla': 'Corriente\n(paralelo)',
     'A_type': r'$A_i = i_o/i_i$',
     'Zin': r'$Z_{in} \div (1+A\beta)$', 'Zout': r'$Z_{out} \times (1+A\beta)$',
     'color_in': COLOR_RECTA, 'color_out': COLOR_PUNTO}
]

for ax, topo in zip(axes.flat, topologies):
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 8)
    ax.axis('off')

    # Título
    ax.text(5, 7.5, topo['name'], ha='center', va='center', fontsize=13,
            fontweight='bold', color=COLOR_PRINCIPAL)

    # Tipo de A
    ax.text(5, 6.5, topo['A_type'], ha='center', va='center', fontsize=12)

    # Mezcla y muestreo
    ax.text(1.5, 5, f"Mezcla:\n{topo['mezcla']}", ha='center', va='center',
            fontsize=10, bbox=dict(boxstyle='round', facecolor='#e8f4fd', edgecolor='gray'))
    ax.text(8.5, 5, f"Muestreo:\n{topo['muestreo']}", ha='center', va='center',
            fontsize=10, bbox=dict(boxstyle='round', facecolor='#fde8e8', edgecolor='gray'))

    # Impedancias
    ax.text(2.5, 2.5, r'$Z_{in}$:', ha='center', va='center', fontsize=12, fontweight='bold')
    ax.text(2.5, 1.5, topo['Zin'], ha='center', va='center', fontsize=11,
            color=topo['color_in'],
            bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor=topo['color_in']))

    ax.text(7.5, 2.5, r'$Z_{out}$:', ha='center', va='center', fontsize=12, fontweight='bold')
    ax.text(7.5, 1.5, topo['Zout'], ha='center', va='center', fontsize=11,
            color=topo['color_out'],
            bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor=topo['color_out']))

    # Flecha central
    ax.annotate('', xy=(6.5, 5), xytext=(3.5, 5),
                arrowprops=dict(arrowstyle='->', linewidth=2, color='gray'))
    ax.text(5, 5, r'$A$', ha='center', va='center', fontsize=14, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', edgecolor=COLOR_PRINCIPAL, linewidth=2))

fig.suptitle('Las 4 Topologías de Realimentación', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

---

## 7. Estabilidad

### 7.1 Condición de estabilidad

Un sistema realimentado es **inestable** si existe una frecuencia $\omega_0$ tal que:

$$|A(j\omega_0)\beta| = 1 \quad \text{y} \quad \angle A(j\omega_0)\beta = -180°$$

Es decir, si la señal de realimentación llega **en fase** con la señal de entrada y con la **misma amplitud**, se produce oscilación (criterio de Barkhausen).

### 7.2 Margen de fase (Phase Margin, PM)

Es la "distancia angular" a la inestabilidad cuando $|A\beta| = 1$ (0 dB):

$$\boxed{PM = 180° + \angle A(j\omega_{gc})\beta}$$

donde $\omega_{gc}$ es la frecuencia de cruce de ganancia ($|A\beta| = 0$ dB).

- $PM > 0$: sistema estable
- $PM > 45°$: recomendado para buena respuesta transitoria
- $PM \approx 60°$: óptimo en muchas aplicaciones

### 7.3 Margen de ganancia (Gain Margin, GM)

Es cuántos dB por debajo de 0 dB está $|A\beta|$ cuando la fase es $-180°$:

$$\boxed{GM = -20\log_{10}|A(j\omega_{pc})\beta|}$$

donde $\omega_{pc}$ es la frecuencia de cruce de fase ($\angle A\beta = -180°$).

- $GM > 0$ dB: sistema estable
- $GM > 10$ dB: recomendado

In [ ]:
# Diagrama de Bode con margen de fase y margen de ganancia
# Sistema con 3 polos: demuestra cómo la fase puede llegar a -180°

A0 = 1e4
beta = 0.01
fp1 = 1e3
fp2 = 1e5
fp3 = 1e6

f = np.logspace(1, 8, 2000)
w = 2 * np.pi * f
s = 1j * w

# Ganancia de lazo A*beta con 3 polos
wp1, wp2, wp3 = 2*np.pi*fp1, 2*np.pi*fp2, 2*np.pi*fp3
A_s = A0 / ((1 + s/wp1) * (1 + s/wp2) * (1 + s/wp3))
T = A_s * beta   # loop gain

T_mag_dB = 20 * np.log10(np.abs(T))
T_phase = np.degrees(np.unwrap(np.angle(T)))

# Encontrar frecuencia de cruce de ganancia (|T| = 0 dB)
idx_gc = np.argmin(np.abs(T_mag_dB))
f_gc = f[idx_gc]
phase_at_gc = T_phase[idx_gc]
PM = 180 + phase_at_gc

# Encontrar frecuencia de cruce de fase (phase = -180°)
idx_pc = np.argmin(np.abs(T_phase + 180))
f_pc = f[idx_pc]
mag_at_pc = T_mag_dB[idx_pc]
GM = -mag_at_pc

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 9), sharex=True)

# --- Magnitud ---
ax1.semilogx(f, T_mag_dB, color=COLOR_BETA, linewidth=2.5, label=r'$|A\beta|$')
ax1.axhline(y=0, color='black', linewidth=1, linestyle='-')

# Marcar cruce de ganancia
ax1.axvline(x=f_gc, color=COLOR_CL, linestyle=':', linewidth=1.5, alpha=0.7)
ax1.plot(f_gc, 0, 'o', color=COLOR_CL, markersize=10, zorder=5)
ax1.annotate(f'$f_{{gc}}$ = {f_gc/1e3:.0f} kHz',
             xy=(f_gc, 0), xytext=(f_gc*3, 10), fontsize=11,
             arrowprops=dict(arrowstyle='->', color=COLOR_CL), color=COLOR_CL)

# Marcar cruce de fase (para GM)
ax1.axvline(x=f_pc, color=COLOR_RECTA, linestyle=':', linewidth=1.5, alpha=0.7)
ax1.plot(f_pc, mag_at_pc, 's', color=COLOR_RECTA, markersize=10, zorder=5)
ax1.annotate(f'GM = {GM:.1f} dB',
             xy=(f_pc, mag_at_pc), xytext=(f_pc*0.1, mag_at_pc+5), fontsize=12,
             arrowprops=dict(arrowstyle='->', color=COLOR_RECTA),
             color=COLOR_RECTA, fontweight='bold',
             bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor=COLOR_RECTA))
# Flecha vertical para GM
ax1.annotate('', xy=(f_pc, 0), xytext=(f_pc, mag_at_pc),
             arrowprops=dict(arrowstyle='<->', color=COLOR_RECTA, linewidth=2))

ax1.set_ylabel('Magnitud (dB)')
ax1.set_title(r'Diagrama de Bode de $A\beta$ - Márgenes de Estabilidad', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, which='both', alpha=0.3)
ax1.set_ylim(-60, 50)

# --- Fase ---
ax2.semilogx(f, T_phase, color=COLOR_BETA, linewidth=2.5, label=r'$\angle A\beta$')
ax2.axhline(y=-180, color='black', linewidth=1, linestyle='-')

# Marcar PM
ax2.axvline(x=f_gc, color=COLOR_CL, linestyle=':', linewidth=1.5, alpha=0.7)
ax2.plot(f_gc, phase_at_gc, 'o', color=COLOR_CL, markersize=10, zorder=5)
ax2.annotate(f'PM = {PM:.1f}\u00b0',
             xy=(f_gc, phase_at_gc), xytext=(f_gc*3, phase_at_gc+20), fontsize=12,
             arrowprops=dict(arrowstyle='->', color=COLOR_CL),
             color=COLOR_CL, fontweight='bold',
             bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor=COLOR_CL))
# Flecha vertical para PM
ax2.annotate('', xy=(f_gc, -180), xytext=(f_gc, phase_at_gc),
             arrowprops=dict(arrowstyle='<->', color=COLOR_CL, linewidth=2))

# Marcar cruce de fase
ax2.axvline(x=f_pc, color=COLOR_RECTA, linestyle=':', linewidth=1.5, alpha=0.7)
ax2.plot(f_pc, -180, 's', color=COLOR_RECTA, markersize=10, zorder=5)

ax2.set_xlabel('Frecuencia (Hz)')
ax2.set_ylabel('Fase (grados)')
ax2.legend(fontsize=11)
ax2.set_ylim(-300, 0)
ax2.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nResultados de estabilidad:")
print(f"  Frecuencia de cruce de ganancia: f_gc = {f_gc/1e3:.1f} kHz")
print(f"  Fase en f_gc: {phase_at_gc:.1f}\u00b0")
print(f"  Margen de fase (PM) = {PM:.1f}\u00b0")
print(f"  Frecuencia de cruce de fase: f_pc = {f_pc/1e3:.1f} kHz")
print(f"  |A\u03b2| en f_pc: {mag_at_pc:.1f} dB")
print(f"  Margen de ganancia (GM) = {GM:.1f} dB")

---

# EJERCICIOS RESUELTOS

---

## Ejercicio 1: Ganancia y Sensibilidad Básicas

**Enunciado:** Un amplificador tiene ganancia en lazo abierto $A = 1000$ y se realimenta con $\beta = 0.01$.

Calcular:
1. La ganancia en lazo cerrado $A_{CL}$
2. La sensibilidad (factor de desensibilización)
3. Si $A$ varía un 20%, ¿cuánto varía $A_{CL}$?

In [ ]:
# Ejercicio 1: Ganancia y sensibilidad
print("="*60)
print("EJERCICIO 1: Ganancia y Sensibilidad")
print("="*60)

A = 1000
beta = 0.01

# 1. Ganancia en lazo cerrado
A_beta = A * beta
A_CL = A / (1 + A_beta)
A_ideal = 1 / beta

print(f"\nDatos: A = {A}, \u03b2 = {beta}")
print(f"  A\u03b2 = {A_beta}")
print(f"  1 + A\u03b2 = {1 + A_beta}")

print(f"\n1) Ganancia en lazo cerrado:")
print(f"   A_CL = A/(1+A\u03b2) = {A}/{1+A_beta} = {A_CL:.4f}")
print(f"   Valor ideal (1/\u03b2) = {A_ideal:.0f}")
print(f"   Error = {abs(A_CL - A_ideal)/A_ideal*100:.2f}%")

# 2. Factor de desensibilización
S = 1 / (1 + A_beta)
print(f"\n2) Sensibilidad relativa:")
print(f"   S = 1/(1+A\u03b2) = 1/{1+A_beta} = {S:.4f}")
print(f"   Las variaciones relativas de A se reducen {1/S:.0f} veces en A_CL")

# 3. Variación del 20%
delta_A_rel = 0.20  # 20%
delta_ACL_rel = S * delta_A_rel
A_new = A * (1 + delta_A_rel)
A_CL_new = A_new / (1 + A_new * beta)
delta_ACL_exact = (A_CL_new - A_CL) / A_CL

print(f"\n3) Si A varía un 20%:")
print(f"   Variación relativa de A_CL \u2248 S \u00d7 20% = {S:.4f} \u00d7 20% = {delta_ACL_rel*100:.3f}%")
print(f"   Verificación exacta:")
print(f"     A_new = {A_new:.0f}")
print(f"     A_CL_new = {A_CL_new:.4f}")
print(f"     Variación real = {delta_ACL_exact*100:.3f}%")

print(f"\n  \u2192 La realimentación reduce la variación de {delta_A_rel*100:.0f}% a {delta_ACL_exact*100:.3f}%")
print(f"    (factor de reducción \u2248 {delta_A_rel/abs(delta_ACL_exact):.1f}\u00d7)")

---

## Ejercicio 2: Ancho de Banda y GBW

**Enunciado:** Un amplificador tiene las siguientes características en lazo abierto:
- Ganancia DC: $A_0 = 10\,000$
- Polo dominante: $f_{p1} = 1$ kHz
- Se aplica realimentación con $\beta = 0.1$

Calcular:
1. La ganancia en lazo cerrado $A_{CL,0}$
2. La nueva frecuencia del polo $f_{CL,p1}$
3. El producto ganancia-ancho de banda (GBW) en ambos casos

In [ ]:
# Ejercicio 2: Ancho de banda y GBW
print("="*60)
print("EJERCICIO 2: Ancho de Banda y GBW")
print("="*60)

A0 = 10000
fp1 = 1e3   # Hz
beta = 0.1

A0_beta = A0 * beta
factor = 1 + A0_beta

print(f"\nDatos: A0 = {A0}, fp1 = {fp1/1e3:.0f} kHz, \u03b2 = {beta}")
print(f"  A0\u03b2 = {A0_beta:.0f}")
print(f"  1 + A0\u03b2 = {factor:.0f}")

# 1. Ganancia CL
A_CL0 = A0 / factor
print(f"\n1) Ganancia en lazo cerrado:")
print(f"   A_CL,0 = A0/(1+A0\u03b2) = {A0}/{factor:.0f} = {A_CL0:.4f}")
print(f"   En dB: {20*np.log10(A_CL0):.2f} dB")

# 2. Nuevo polo
f_CL_p1 = factor * fp1
print(f"\n2) Nuevo polo:")
print(f"   f_CL,p1 = (1+A0\u03b2) \u00d7 fp1 = {factor:.0f} \u00d7 {fp1/1e3:.0f} kHz = {f_CL_p1/1e6:.1f} MHz")

# 3. GBW
GBW_OL = A0 * fp1
GBW_CL = A_CL0 * f_CL_p1

print(f"\n3) Producto Ganancia \u00d7 Ancho de Banda:")
print(f"   GBW_OL = A0 \u00d7 fp1 = {A0} \u00d7 {fp1:.0f} = {GBW_OL:.2e} Hz")
print(f"   GBW_CL = A_CL,0 \u00d7 f_CL,p1 = {A_CL0:.4f} \u00d7 {f_CL_p1:.0f} = {GBW_CL:.2e} Hz")
print(f"   \u2192 GBW se conserva")

# Gráfica
fig, ax = plt.subplots(figsize=(12, 5))
f = np.logspace(1, 8, 1000)
w = 2*np.pi*f
s = 1j*w

A_OL = A0 / (1 + s/(2*np.pi*fp1))
A_CL = A_CL0 / (1 + s/(2*np.pi*f_CL_p1))

ax.semilogx(f, 20*np.log10(np.abs(A_OL)), color=COLOR_PRINCIPAL, linewidth=2.5,
            label=f'Lazo abierto: $A_0$={A0}, $f_{{p1}}$={fp1/1e3:.0f} kHz')
ax.semilogx(f, 20*np.log10(np.abs(A_CL)), color=COLOR_CL, linewidth=2.5,
            label=f'Lazo cerrado: $A_{{CL,0}}$={A_CL0:.2f}, $f_{{CL,p1}}$={f_CL_p1/1e6:.1f} MHz')
ax.axhline(y=0, color='gray', linewidth=0.8, linestyle=':')

# Marcar BW
for fx, col in [(fp1, COLOR_PRINCIPAL), (f_CL_p1, COLOR_CL)]:
    ax.axvline(x=fx, color=col, linestyle=':', alpha=0.5, linewidth=1.5)

# GBW line
f_gbw = np.logspace(np.log10(fp1), 8, 100)
gbw_line = 20*np.log10(GBW_OL / f_gbw)
ax.semilogx(f_gbw, gbw_line, 'k--', alpha=0.3, linewidth=1.5, label=f'GBW = {GBW_OL:.0e} Hz')

ax.set_xlabel('Frecuencia (Hz)')
ax.set_ylabel('Magnitud (dB)')
ax.set_title('Ejercicio 2: Efecto de la realimentación en la respuesta en frecuencia')
ax.legend(fontsize=10)
ax.set_ylim(-20, 90)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

---

## Ejercicio 3: Identificación de Topología

**Enunciado:** Dado el siguiente circuito de amplificador operacional en configuración no inversora, identificar:

1. Tipo de muestreo (tensión o corriente)
2. Tipo de mezcla (serie o paralelo)
3. Topología de realimentación
4. Factor $\beta$
5. Efecto esperado sobre $Z_{in}$ y $Z_{out}$

El circuito tiene:
- Op-amp con $v^+$ conectada a $v_{in}$
- $R_1 = 1\,k\Omega$ entre la salida y $v^-$
- $R_2 = 9\,k\Omega$ entre $v^-$ y tierra

In [ ]:
# Ejercicio 3: Identificación de topología
print("="*60)
print("EJERCICIO 3: Identificación de Topología de Realimentación")
print("="*60)

R1 = 1e3    # entre salida y v-
R2 = 9e3    # entre v- y GND

print(f"\nCircuito: Amplificador no inversor")
print(f"  R1 (salida \u2192 v\u207b) = {R1/1e3:.0f} k\u03a9")
print(f"  R2 (v\u207b \u2192 GND) = {R2/1e3:.0f} k\u03a9")

print(f"\n1) Tipo de muestreo:")
print(f"   La red \u03b2 se conecta entre la SALIDA y v\u207b")
print(f"   \u2192 Muestrea la TENSIÓN de salida (conexión en paralelo)")

print(f"\n2) Tipo de mezcla:")
print(f"   La señal de error (v\u207a - v\u207b) se aplica como diferencia de tensiones")
print(f"   La entrada vin y la realimentación se conectan a terminales diferentes")
print(f"   \u2192 Mezcla en SERIE (la señal de feedback se resta en tensión)")

print(f"\n3) Topología:")
print(f"   Muestreo de tensión + Mezcla en serie")
print(f"   \u2192 SERIE-PARALELO (Tensión-Tensión)")

beta = R2 / (R1 + R2)
A_CL_ideal = 1 / beta
A_CL_formula = (R1 + R2) / R2

print(f"\n4) Factor \u03b2:")
print(f"   \u03b2 = R2/(R1+R2) = {R2/1e3:.0f}/({R1/1e3:.0f}+{R2/1e3:.0f}) = {beta:.4f}")
print(f"   A_CL \u2248 1/\u03b2 = (R1+R2)/R2 = {A_CL_formula:.2f}")

print(f"\n5) Efecto sobre impedancias:")
print(f"   Z_in,CL = Z_in \u00d7 (1+A\u03b2)  \u2192 AUMENTA")
print(f"   Z_out,CL = Z_out \u00f7 (1+A\u03b2) \u2192 DISMINUYE")
print(f"   \u2192 Ideal como buffer de tensión (alta Zin, baja Zout)")

# Dibujar circuito con schemdraw
with schemdraw.Drawing(show=True) as d:
    d.config(fontsize=14)

    # Op-amp
    op = d.add(elm.Opamp().right().label('A', loc='center', fontsize=12))

    # Entrada
    d.add(elm.Line().left().at(op.in1).length(1))
    d.add(elm.Dot())
    vin_node = d.here
    d.add(elm.Line().left().length(1))
    d.add(elm.SourceV().down().label('$v_{in}$', loc='left'))
    d.add(elm.Ground())

    # Salida
    d.add(elm.Line().right().at(op.out).length(2))
    d.add(elm.Dot())
    out_node = d.here
    d.add(elm.Line().right().length(0.5))
    d.add(elm.Dot(open=True).label('$v_{out}$', loc='right'))

    # Realimentación: R1 desde salida hasta v-
    d.add(elm.Line().down().at(out_node).length(2))
    d.add(elm.Resistor().left().label('$R_1 = 1\\,k\\Omega$', loc='top'))
    d.add(elm.Dot())
    fb_node = d.here

    # Conexión a v-
    d.add(elm.Line().up().at(fb_node).toy(op.in2))
    d.add(elm.Line().right().tox(op.in2))

    # R2 a tierra
    d.add(elm.Resistor().down().at(fb_node).label('$R_2 = 9\\,k\\Omega$', loc='left'))
    d.add(elm.Ground())

---

## Ejercicio 4: Reducción de Distorsión

**Enunciado:** Un amplificador tiene los siguientes coeficientes no lineales:
- Ganancia lineal: $a_1 = 500$
- Coeficiente de segundo armónico: $HD_2 = 5\%$
- Coeficiente de tercer armónico: $HD_3 = 1\%$

Se aplica realimentación con $\beta = 0.02$.

Calcular la distorsión en lazo cerrado para:
1. Misma señal de entrada
2. Misma señal de salida

In [ ]:
# Ejercicio 4: Reducción de distorsión
print("="*60)
print("EJERCICIO 4: Reducción de Distorsión")
print("="*60)

a1 = 500
HD2 = 5.0    # %
HD3 = 1.0    # %
beta = 0.02

a1_beta = a1 * beta
factor = 1 + a1_beta

print(f"\nDatos: a1 = {a1}, HD2 = {HD2}%, HD3 = {HD3}%, \u03b2 = {beta}")
print(f"  a1\u00b7\u03b2 = {a1_beta:.0f}")
print(f"  1 + a1\u00b7\u03b2 = {factor:.0f}")

print(f"\n{'='*50}")
print(f"1) MISMA SEÑAL DE ENTRADA")
print(f"{'='*50}")
print(f"   HD_CL,i \u2248 HD_i / (1+a1\u00b7\u03b2)^i")
HD2_CL_in = HD2 / factor**2
HD3_CL_in = HD3 / factor**3
print(f"\n   HD2_CL = {HD2}% / {factor}\u00b2 = {HD2}% / {factor**2:.0f} = {HD2_CL_in:.5f}%")
print(f"   HD3_CL = {HD3}% / {factor}\u00b3 = {HD3}% / {factor**3:.0f} = {HD3_CL_in:.7f}%")
print(f"\n   Reducción HD2: {HD2/HD2_CL_in:.0f}\u00d7 ({factor**2:.0f}\u00d7)")
print(f"   Reducción HD3: {HD3/HD3_CL_in:.0f}\u00d7 ({factor**3:.0f}\u00d7)")

print(f"\n{'='*50}")
print(f"2) MISMA SEÑAL DE SALIDA")
print(f"{'='*50}")
print(f"   HD_CL,i \u2248 HD_i / (1+a1\u00b7\u03b2)")
HD2_CL_out = HD2 / factor
HD3_CL_out = HD3 / factor
print(f"\n   HD2_CL = {HD2}% / {factor} = {HD2_CL_out:.4f}%")
print(f"   HD3_CL = {HD3}% / {factor} = {HD3_CL_out:.4f}%")
print(f"\n   Reducción HD2: {factor:.0f}\u00d7")
print(f"   Reducción HD3: {factor:.0f}\u00d7")

# Gráfica comparativa
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

labels = ['HD\u2082', 'HD\u2083']
original = [HD2, HD3]
misma_entrada = [HD2_CL_in, HD3_CL_in]
misma_salida = [HD2_CL_out, HD3_CL_out]

x = np.arange(len(labels))
width = 0.25

# Gráfica misma entrada (escala log)
bars1 = ax1.bar(x - width, original, width, label='Sin realimentación',
                color=COLOR_RECTA, alpha=0.8)
bars2 = ax1.bar(x, misma_entrada, width, label='Con realim. (misma entrada)',
                color=COLOR_CL, alpha=0.8)
bars3 = ax1.bar(x + width, misma_salida, width, label='Con realim. (misma salida)',
                color=COLOR_PUNTO, alpha=0.8)

ax1.set_yscale('log')
ax1.set_ylabel('Distorsión armónica (%)')
ax1.set_title('Comparación de distorsión')
ax1.set_xticks(x)
ax1.set_xticklabels(labels)
ax1.legend(fontsize=9)
ax1.grid(True, which='both', alpha=0.3, axis='y')

# Anotar valores
for bar, val in zip(bars1, original):
    ax1.text(bar.get_x() + bar.get_width()/2, val*1.3, f'{val:.1f}%',
             ha='center', va='bottom', fontsize=9)
for bar, val in zip(bars2, misma_entrada):
    ax1.text(bar.get_x() + bar.get_width()/2, val*1.3, f'{val:.4f}%',
             ha='center', va='bottom', fontsize=8)
for bar, val in zip(bars3, misma_salida):
    ax1.text(bar.get_x() + bar.get_width()/2, val*1.3, f'{val:.3f}%',
             ha='center', va='bottom', fontsize=8)

# Tabla
ax2.axis('off')
table_data = [
    ['', 'Sin FB', 'Misma entrada', 'Misma salida'],
    ['HD\u2082', f'{HD2:.1f}%', f'{HD2_CL_in:.5f}%', f'{HD2_CL_out:.4f}%'],
    ['HD\u2083', f'{HD3:.1f}%', f'{HD3_CL_in:.7f}%', f'{HD3_CL_out:.4f}%'],
    ['Factor', '1\u00d7',
     f'HD\u2082: {factor**2:.0f}\u00d7\nHD\u2083: {factor**3:.0f}\u00d7',
     f'{factor:.0f}\u00d7'],
]
table = ax2.table(cellText=table_data, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 2)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#d4e6f1')
        cell.set_text_props(fontweight='bold')
    if col == 0 and row > 0:
        cell.set_text_props(fontweight='bold')
ax2.set_title('Tabla resumen', fontsize=13, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

---

## Ejercicio 5 (Numérico): Desensibilización con $A\beta = 50$

**Enunciado:** Un amplificador con $A = 5000$ y $\beta = 0.01$ experimenta una variación de $A$ del $\pm 10\%$ debido a temperatura.

Calcular la variación de $A_{CL}$ y compararla con el caso sin realimentación.

In [ ]:
# Ejercicio 5: Desensibilización con A*beta = 50
print("="*60)
print("EJERCICIO 5: Desensibilización con A\u03b2 = 50")
print("="*60)

A = 5000
beta = 0.01
delta_A_percent = 10  # +/-10%

A_beta = A * beta
factor = 1 + A_beta

print(f"\nDatos: A = {A}, \u03b2 = {beta}")
print(f"  A\u03b2 = {A_beta:.0f}")
print(f"  1 + A\u03b2 = {factor:.0f}")

# Ganancia nominal
A_CL = A / factor
print(f"\nGanancia nominal:")
print(f"  A_CL = {A}/{factor:.0f} = {A_CL:.4f}")
print(f"  1/\u03b2 = {1/beta:.0f}")

# Con variación de +/-10%
A_plus = A * 1.10
A_minus = A * 0.90
A_CL_plus = A_plus / (1 + A_plus * beta)
A_CL_minus = A_minus / (1 + A_minus * beta)

delta_ACL_plus = (A_CL_plus - A_CL) / A_CL * 100
delta_ACL_minus = (A_CL_minus - A_CL) / A_CL * 100

print(f"\nVariación de A = \u00b1{delta_A_percent}%:")
print(f"  A+10% = {A_plus:.0f} \u2192 A_CL = {A_CL_plus:.4f} ({delta_ACL_plus:+.4f}%)")
print(f"  A-10% = {A_minus:.0f} \u2192 A_CL = {A_CL_minus:.4f} ({delta_ACL_minus:+.4f}%)")

# Factor de desensibilización
S_abs = 1 / factor**2
S_rel = 1 / factor
print(f"\nFactores de desensibilización:")
print(f"  Sensibilidad absoluta: dA_CL/dA = 1/(1+A\u03b2)\u00b2 = 1/{factor**2:.0f} = {S_abs:.6f}")
print(f"  Sensibilidad relativa: dA_CL/A_CL / (dA/A) = 1/(1+A\u03b2) = 1/{factor:.0f} = {S_rel:.5f}")
print(f"\n  \u2192 Variación absoluta reducida {factor**2:.0f}\u00d7 (= (1+A\u03b2)\u00b2)")
print(f"  \u2192 Variación relativa reducida {factor:.0f}\u00d7 (= 1+A\u03b2)")

# Gráfica: variación de A_CL con A
fig, ax = plt.subplots(figsize=(12, 5))

A_range = np.linspace(A*0.7, A*1.3, 200)
A_CL_range = A_range / (1 + A_range * beta)

ax.plot(A_range, A_CL_range, color=COLOR_CL, linewidth=2.5, label=r'$A_{CL}(A)$')
ax.axhline(y=1/beta, color=COLOR_RECTA, linestyle='--', linewidth=1.5,
           label=rf'$1/\beta = {1/beta:.0f}$')
ax.axhline(y=A_CL, color='gray', linestyle=':', alpha=0.5)

# Zona de variación
ax.axvspan(A_minus, A_plus, alpha=0.15, color=COLOR_PRINCIPAL,
           label=f'A \u00b1 {delta_A_percent}%')
ax.axhspan(A_CL_minus, A_CL_plus, alpha=0.15, color=COLOR_CL,
           label=f'A_CL \u00b1 {abs(delta_ACL_plus):.3f}%')

ax.plot(A, A_CL, 'o', color=COLOR_PUNTO, markersize=10, zorder=5, label='Punto nominal')
ax.plot([A_plus, A_minus], [A_CL_plus, A_CL_minus], 's',
        color=COLOR_RECTA, markersize=8, zorder=5)

ax.set_xlabel(r'Ganancia en lazo abierto $A$')
ax.set_ylabel(r'Ganancia en lazo cerrado $A_{CL}$')
ax.set_title(rf'Desensibilización: $A\beta = {A_beta:.0f}$, variación de $A$ = \u00b1{delta_A_percent}%')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

# CATÁLOGO DE EJERCICIOS

## Tipo 1: Ganancia en lazo cerrado

**Método:**
1. Identificar $A$ y $\beta$ del circuito
2. Calcular $A\beta$ (ganancia de lazo)
3. Aplicar $A_{CL} = A/(1+A\beta)$
4. Verificar si $A\beta \gg 1$ para usar la aproximación $A_{CL} \approx 1/\beta$

**Variantes:**
- Calcular $A_{CL}$ dado $A$ y $\beta$
- Encontrar $\beta$ para obtener una $A_{CL}$ deseada
- Determinar el $A$ mínimo para que el error sea menor que un porcentaje dado

---

## Tipo 2: Desensibilización

**Método:**
1. Calcular $1 + A\beta$
2. Sensibilidad relativa: $\Delta A_{CL}/A_{CL} = \Delta A/(A \cdot (1+A\beta))$
3. Sensibilidad absoluta: $\Delta A_{CL} = \Delta A/(1+A\beta)^2$

**Variantes:**
- Calcular variación de $A_{CL}$ dada variación de $A$
- Diseñar $A\beta$ para limitar la sensibilidad a un valor máximo
- Comparar variación con y sin realimentación

---

## Tipo 3: Ancho de banda y GBW

**Método:**
1. Identificar $A_0$ y $\omega_{p1}$ del amplificador en lazo abierto
2. Calcular $GBW = A_0 \cdot \omega_{p1}$
3. Nuevo polo: $\omega_{CL,p1} = (1+A_0\beta)\omega_{p1}$
4. Verificar conservación del GBW

**Variantes:**
- Calcular nuevo BW con realimentación
- Diseñar $\beta$ para lograr un BW mínimo
- Sistemas con múltiples polos

---

## Tipo 4: Identificación de topología

**Método:**
1. Identificar cómo se muestrea la salida:
   - ¿Se toma tensión (paralelo) o corriente (serie)?
2. Identificar cómo se mezcla con la entrada:
   - ¿Se suma/resta en serie (tensión) o en paralelo (corriente)?
3. Determinar la topología y sus efectos sobre $Z_{in}$, $Z_{out}$
4. Calcular $\beta$ de la red de realimentación

**Variantes:**
- Amplificador no inversor (serie-paralelo)
- Amplificador de transconductancia con degeneración (serie-serie)
- Amplificador de transimpedancia (paralelo-paralelo)

---

## Tipo 5: Estabilidad

**Método:**
1. Obtener $A(j\omega)\beta$ (Bode de la ganancia de lazo)
2. Encontrar $\omega_{gc}$: frecuencia donde $|A\beta| = 0$ dB
3. PM = $180° + \angle A(j\omega_{gc})\beta$
4. Encontrar $\omega_{pc}$: frecuencia donde $\angle A\beta = -180°$
5. GM = $-20\log|A(j\omega_{pc})\beta|$

**Variantes:**
- Verificar estabilidad de un sistema dado
- Diseñar compensación para lograr PM y GM deseados
- Determinar $\beta_{max}$ para estabilidad

---

# RESUMEN

## Tabla resumen de realimentación negativa

| Propiedad | Sin realimentación | Con realimentación | Factor |
|-----------|-------------------|-------------------|--------|
| **Ganancia** | $A$ | $\frac{A}{1+A\beta} \approx \frac{1}{\beta}$ | $\div (1+A\beta)$ |
| **Sensibilidad** | $\frac{dA}{A}$ | $\frac{1}{1+A\beta}\frac{dA}{A}$ | $\div (1+A\beta)$ |
| **Ancho de banda** | $\omega_{p1}$ | $(1+A_0\beta)\omega_{p1}$ | $\times (1+A\beta)$ |
| **GBW** | $A_0 \omega_{p1}$ | $A_0 \omega_{p1}$ | $= 1$ (conservado) |
| **Distorsión (misma $x$)** | $HD_i$ | $\frac{HD_i}{(1+a_1\beta)^i}$ | $\div (1+a_1\beta)^i$ |
| **Distorsión (misma $y$)** | $HD_i$ | $\frac{HD_i}{1+a_1\beta}$ | $\div (1+a_1\beta)$ |

## Topologías y su efecto en impedancias

| Topología | $A$ es | Mezcla | Muestreo | $Z_{in}$ | $Z_{out}$ |
|-----------|--------|--------|----------|-----------|------------|
| Serie-Paralelo | $A_v$ | Serie | Paralelo (V) | $\uparrow (1+A\beta)$ | $\downarrow (1+A\beta)$ |
| Serie-Serie | $G_m$ | Serie | Serie (I) | $\uparrow (1+A\beta)$ | $\uparrow (1+A\beta)$ |
| Paralelo-Paralelo | $R_m$ | Paralelo | Paralelo (V) | $\downarrow (1+A\beta)$ | $\downarrow (1+A\beta)$ |
| Paralelo-Serie | $A_i$ | Paralelo | Serie (I) | $\downarrow (1+A\beta)$ | $\uparrow (1+A\beta)$ |

## Regla mnemotécnica para impedancias

- **Serie en la entrada** $\rightarrow$ $Z_{in}$ **aumenta** $\times (1+A\beta)$
- **Paralelo en la entrada** $\rightarrow$ $Z_{in}$ **disminuye** $\div (1+A\beta)$
- **Paralelo en la salida** (muestreo de V) $\rightarrow$ $Z_{out}$ **disminuye** $\div (1+A\beta)$
- **Serie en la salida** (muestreo de I) $\rightarrow$ $Z_{out}$ **aumenta** $\times (1+A\beta)$

## Criterios de estabilidad

| Margen | Definición | Recomendado |
|--------|-----------|-------------|
| **Phase Margin** | $PM = 180° + \angle A\beta\big|_{|A\beta|=0\text{dB}}$ | $PM > 45°$ |
| **Gain Margin** | $GM = -20\log|A\beta|\big|_{\angle A\beta = -180°}$ | $GM > 10$ dB |

In [ ]:
# Ejemplo completo: Diseño de un amplificador con realimentación
# Comparación de todas las propiedades

print("="*60)
print("EJEMPLO INTEGRADOR: Diseño de amplificador realimentado")
print("="*60)

# Parámetros del amplificador en lazo abierto
A0 = 50000
fp1 = 500       # Hz (polo dominante)
HD2_OL = 3.0    # %
HD3_OL = 0.5    # %
Zin_OL = 10e3   # 10 kOhm
Zout_OL = 1e3   # 1 kOhm

# Especificación: A_CL ~ 50 -> beta = 1/50 = 0.02
beta = 0.02
A_beta = A0 * beta
factor = 1 + A_beta

print(f"\nAmplificador en lazo abierto:")
print(f"  A0 = {A0}, fp1 = {fp1} Hz")
print(f"  HD2 = {HD2_OL}%, HD3 = {HD3_OL}%")
print(f"  Zin = {Zin_OL/1e3:.0f} k\u03a9, Zout = {Zout_OL/1e3:.0f} k\u03a9")

print(f"\nDiseño: \u03b2 = {beta} (topología serie-paralelo)")
print(f"  A\u03b2 = {A_beta:.0f}")
print(f"  1 + A\u03b2 = {factor:.0f}")

# Cálculos
A_CL = A0 / factor
f_CL = factor * fp1
GBW = A0 * fp1
HD2_CL = HD2_OL / factor
HD3_CL = HD3_OL / factor
Zin_CL = Zin_OL * factor
Zout_CL = Zout_OL / factor

print(f"\nResultados (lazo cerrado, serie-paralelo):")
print(f"{'Propiedad':<20} {'Lazo abierto':>15} {'Lazo cerrado':>15} {'Factor':>12}")
print("-"*65)
print(f"{'Ganancia':<20} {A0:>15.0f} {A_CL:>15.2f} {'\u00f7'+str(factor):>12}")
print(f"{'BW':<20} {str(fp1)+' Hz':>15} {str(round(f_CL))+' Hz':>15} {'\u00d7'+str(factor):>12}")
print(f"{'GBW':<20} {str(round(GBW/1e6))+' MHz':>15} {str(round(A_CL*f_CL/1e6))+' MHz':>15} {'=':>12}")
print(f"{'HD2 (misma sal.)':<20} {str(HD2_OL)+'%':>15} {f'{HD2_CL:.4f}%':>15} {'\u00f7'+str(factor):>12}")
print(f"{'HD3 (misma sal.)':<20} {str(HD3_OL)+'%':>15} {f'{HD3_CL:.5f}%':>15} {'\u00f7'+str(factor):>12}")
print(f"{'Zin':<20} {str(round(Zin_OL/1e3))+' k\u03a9':>15} {f'{Zin_CL/1e6:.1f} M\u03a9':>15} {'\u00d7'+str(factor):>12}")
print(f"{'Zout':<20} {str(round(Zout_OL/1e3))+' k\u03a9':>15} {f'{Zout_CL:.2f} \u03a9':>15} {'\u00f7'+str(factor):>12}")

# Gráfica resumen
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Bode magnitud
ax = axes[0, 0]
f = np.logspace(0, 8, 1000)
w = 2*np.pi*f
s = 1j*w
A_OL_s = A0 / (1 + s/(2*np.pi*fp1))
A_CL_s = A_CL / (1 + s/(2*np.pi*f_CL))

ax.semilogx(f, 20*np.log10(np.abs(A_OL_s)), color=COLOR_PRINCIPAL, linewidth=2, label='Lazo abierto')
ax.semilogx(f, 20*np.log10(np.abs(A_CL_s)), color=COLOR_CL, linewidth=2, label='Lazo cerrado')
ax.axhline(y=0, color='gray', linewidth=0.5, linestyle=':')
ax.set_xlabel('Frecuencia (Hz)')
ax.set_ylabel('|A| (dB)')
ax.set_title('Respuesta en frecuencia')
ax.legend(fontsize=9)
ax.grid(True, which='both', alpha=0.3)

# 2. Desensibilización
ax = axes[0, 1]
A_var = np.linspace(A0*0.5, A0*1.5, 200)
ACL_var = A_var / (1 + A_var * beta)
ax.plot(A_var/A0*100, ACL_var/A_CL*100, color=COLOR_CL, linewidth=2.5,
        label='Con realimentación')
ax.plot([50, 150], [50, 150], color=COLOR_RECTA, linewidth=2, linestyle='--',
        label='Sin realimentación', alpha=0.7)
ax.axhline(y=100, color='gray', linewidth=0.5, linestyle=':')
ax.axvline(x=100, color='gray', linewidth=0.5, linestyle=':')
ax.set_xlabel('A relativa (%)')
ax.set_ylabel('A_CL relativa (%)')
ax.set_title('Desensibilización')
ax.legend(fontsize=9)
ax.set_xlim(50, 150)
ax.set_ylim(90, 110)
ax.grid(True, alpha=0.3)

# 3. Distorsión
ax = axes[1, 0]
armonicos = ['HD\u2082', 'HD\u2083']
sin_fb = [HD2_OL, HD3_OL]
con_fb = [HD2_CL, HD3_CL]
x_pos = np.arange(len(armonicos))
width = 0.35
ax.bar(x_pos - width/2, sin_fb, width, label='Sin FB', color=COLOR_RECTA, alpha=0.8)
ax.bar(x_pos + width/2, con_fb, width, label='Con FB', color=COLOR_CL, alpha=0.8)
for i, (v1, v2) in enumerate(zip(sin_fb, con_fb)):
    ax.text(i-width/2, v1+0.05, f'{v1:.1f}%', ha='center', fontsize=10)
    ax.text(i+width/2, v2+0.005, f'{v2:.4f}%', ha='center', fontsize=9)
ax.set_xticks(x_pos)
ax.set_xticklabels(armonicos)
ax.set_ylabel('Distorsión (%)')
ax.set_title('Reducción de distorsión (misma salida)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

# 4. Impedancias
ax = axes[1, 1]
params = ['Zin', 'Zout']
sin_fb_z = [Zin_OL, Zout_OL]
con_fb_z = [Zin_CL, Zout_CL]
x_pos = np.arange(len(params))
bars1 = ax.bar(x_pos - width/2, sin_fb_z, width, label='Sin FB', color=COLOR_RECTA, alpha=0.8)
bars2 = ax.bar(x_pos + width/2, con_fb_z, width, label='Con FB', color=COLOR_CL, alpha=0.8)
ax.set_yscale('log')
ax.set_xticks(x_pos)
ax.set_xticklabels(params)
ax.set_ylabel('Impedancia (\u03a9)')
ax.set_title('Impedancias (serie-paralelo)')
for bar, val in zip(bars1, sin_fb_z):
    if val >= 1e3:
        label = f'{val/1e3:.0f} k\u03a9'
    else:
        label = f'{val:.0f} \u03a9'
    ax.text(bar.get_x()+bar.get_width()/2, val*1.5, label, ha='center', fontsize=10)
for bar, val in zip(bars2, con_fb_z):
    if val >= 1e6:
        label = f'{val/1e6:.1f} M\u03a9'
    elif val >= 1e3:
        label = f'{val/1e3:.0f} k\u03a9'
    else:
        label = f'{val:.2f} \u03a9'
    ax.text(bar.get_x()+bar.get_width()/2, val*1.5, label, ha='center', fontsize=10)
ax.legend(fontsize=9)
ax.grid(True, which='both', alpha=0.3, axis='y')

fig.suptitle(f'Resumen: Amplificador con \u03b2={beta} (A\u03b2={A_beta:.0f})', fontsize=15, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
# Efecto del factor A*beta sobre todas las propiedades
A0 = 10000
fp1 = 1e3
HD2_base = 5.0  # %

beta_values = np.array([0.001, 0.005, 0.01, 0.02, 0.05, 0.1])
A_beta_values = A0 * beta_values

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Colores para cada beta
cmap = plt.cm.viridis
colors = [cmap(i/len(beta_values)) for i in range(len(beta_values))]

# 1. Ganancia CL vs A*beta
ax = axes[0, 0]
ACL_vals = A0 / (1 + A_beta_values)
ax.bar(range(len(beta_values)), ACL_vals, color=colors, edgecolor='black', linewidth=0.5)
for i, (ab, acl) in enumerate(zip(A_beta_values, ACL_vals)):
    ax.text(i, acl + max(ACL_vals)*0.02, f'{acl:.1f}', ha='center', fontsize=9)
ax.set_xticks(range(len(beta_values)))
ax.set_xticklabels([f'A\u03b2={ab:.0f}' for ab in A_beta_values], rotation=30, fontsize=9)
ax.set_ylabel(r'$A_{CL}$')
ax.set_title('Ganancia en lazo cerrado')
ax.grid(True, alpha=0.3, axis='y')

# 2. BW vs A*beta
ax = axes[0, 1]
BW_vals = (1 + A_beta_values) * fp1
ax.bar(range(len(beta_values)), BW_vals/1e3, color=colors, edgecolor='black', linewidth=0.5)
for i, bw in enumerate(BW_vals):
    if bw >= 1e6:
        label = f'{bw/1e6:.1f} MHz'
    else:
        label = f'{bw/1e3:.0f} kHz'
    ax.text(i, bw/1e3 + max(BW_vals)/1e3*0.02, label, ha='center', fontsize=8)
ax.set_xticks(range(len(beta_values)))
ax.set_xticklabels([f'A\u03b2={ab:.0f}' for ab in A_beta_values], rotation=30, fontsize=9)
ax.set_ylabel('BW (kHz)')
ax.set_title('Ancho de banda')
ax.grid(True, alpha=0.3, axis='y')

# 3. Sensibilidad vs A*beta
ax = axes[1, 0]
sens_vals = 1 / (1 + A_beta_values) * 100
ax.bar(range(len(beta_values)), sens_vals, color=colors, edgecolor='black', linewidth=0.5)
for i, sv in enumerate(sens_vals):
    ax.text(i, sv + max(sens_vals)*0.02, f'{sv:.2f}%', ha='center', fontsize=9)
ax.set_xticks(range(len(beta_values)))
ax.set_xticklabels([f'A\u03b2={ab:.0f}' for ab in A_beta_values], rotation=30, fontsize=9)
ax.set_ylabel('Sensibilidad relativa (%)')
ax.set_title(r'Sensibilidad $\frac{\Delta A_{CL}/A_{CL}}{\Delta A/A}$')
ax.grid(True, alpha=0.3, axis='y')

# 4. HD2 vs A*beta (misma salida)
ax = axes[1, 1]
HD2_vals = HD2_base / (1 + A_beta_values)
ax.bar(range(len(beta_values)), HD2_vals, color=colors, edgecolor='black', linewidth=0.5)
for i, hd in enumerate(HD2_vals):
    ax.text(i, hd + max(HD2_vals)*0.02, f'{hd:.3f}%', ha='center', fontsize=9)
ax.set_xticks(range(len(beta_values)))
ax.set_xticklabels([f'A\u03b2={ab:.0f}' for ab in A_beta_values], rotation=30, fontsize=9)
ax.set_ylabel('HD\u2082 (%)')
ax.set_title('Distorsión HD\u2082 (misma salida)')
ax.grid(True, alpha=0.3, axis='y')

fig.suptitle(r'Efecto de $A\beta$ sobre las propiedades del amplificador ($A_0 = 10000$)',
             fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()